# Baselines — MostPop & ItemKNN on Yambda-50M

All models share one prep (Global Temporal Split, Listen+ positives, dense-id vocab from train), so the numbers are directly comparable. ItemKNN is the bar to beat — paper reports **NDCG@10 = 0.0781** on 50M.

Settings: **Internet On**, GPU not needed (CPU). Optional `HF_TOKEN` secret.

In [ ]:
# --upgrade so re-runs pick up new pushes. [baselines] extra pulls in `implicit`.
!pip install -q --upgrade "ymrec[baselines] @ git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print("No HF_TOKEN secret — anonymous download still works.")

In [ ]:
import time
import numpy as np
import pandas as pd
from ymrec.config import TOPK, PAPER_BASELINES_NDCG10
from ymrec.data.prep import prepare
from ymrec.eval.metrics import evaluate_ranking
from ymrec.baselines.item_knn import fit_recommend

K = max(TOPK)
t0 = time.time()
p = prepare(size="50m")
print(f"prepared in {time.time()-t0:.1f}s | users={p.n_users:,} items={p.n_items:,} "
      f"train_nnz={p.train_ui.nnz:,} eval_users={len(p.eval_user_idx):,}")

In [ ]:
rows = []

def score(name, recs, secs):
    r = evaluate_ranking(recs, p.relevant, n_items=p.n_items, ks=TOPK)
    rows.append({"model": name, "sec": round(secs, 1),
                 **{k: round(v, 4) for k, v in r.items()}})
    print(f"{name:32s} NDCG@10={r['ndcg@10']:.4f}")

# MostPop from the same train matrix (item play-counts).
t = time.time()
pop = np.asarray(p.train_ui.sum(axis=0)).ravel()
order = np.argsort(-pop)
mostpop = np.tile(p.item_ids[order[:K]], (len(p.eval_user_idx), 1))
score("MostPop", mostpop, time.time() - t)

for variant in ["cosine", "tfidf", "bm25"]:
    t = time.time()
    recs = fit_recommend(p.train_ui, p.eval_user_idx, p.item_ids, K,
                         neighbors=100, variant=variant, filter_seen=False)
    score(f"ItemKNN-{variant}", recs, time.time() - t)

# Effect of removing already-heard tracks (expected to HURT — re-listening is real).
t = time.time()
recs = fit_recommend(p.train_ui, p.eval_user_idx, p.item_ids, K,
                     neighbors=100, variant="cosine", filter_seen=True)
score("ItemKNN-cosine(filter_seen)", recs, time.time() - t)

In [ ]:
df = pd.DataFrame(rows).sort_values("ndcg@10", ascending=False).reset_index(drop=True)
print("paper baselines (NDCG@10, 50M):", PAPER_BASELINES_NDCG10["50m"])
df